# SAFIRA | Seção 4.2.1: Exploração de dados

**Inteli, Turma T28, Grupo 01** | Parceiro: **Azul Linhas Aéreas Brasileiras**

Este notebook contém a implementação da exploração de dados documentada na seção 4.2.1 de `documents/documentacao.md`. O objetivo é caracterizar a estrutura, a qualidade e a representatividade das bases fornecidas pela Azul antes da etapa de preparação, conforme a fase de *Data Understanding* da metodologia CRISP-DM (CHAPMAN et al., 2000).

A unidade de análise adotada é **o voo avaliado**, e não o Cliente. O alvo é a detração no NPS principal, binarizada em Detrator contra não-Detrator.

Cada figura deste notebook nasce da execução da célula imediatamente acima dela. Nenhuma imagem é incorporada como arquivo pronto.

> ### Aviso sobre os dados
>
> As bases da Azul **não são versionadas neste repositório**, por restrição do Termo de Abertura de Projeto de Inovação. O diretório `data/` está no `.gitignore` e nenhum arquivo `.csv`, `.pkl` ou `.parquet` é publicado.
>
> Para reproduzir, aponte `CAMINHO_DADOS` na célula de configuração para a pasta local que contém os cinco arquivos, ou monte o Google Drive no Colab. As saídas de célula que exibiriam linhas individuais de Cliente foram mantidas apenas em forma agregada.

---

## 1. Configuração do ambiente

A stack de visualização combina as duas bibliotecas em divisão de responsabilidades deliberada, e não por redundância:

- **`seaborn`** responde pela camada estatística e pelo tema: `barplot`, `lineplot`, `heatmap` e `relplot` já agregam, ordenam categorias e distribuem os pequenos múltiplos, e `set_theme` fixa a paleta institucional em todas as figuras de uma só vez.
- **`matplotlib`** responde pelo que o `seaborn` não abstrai: eixos secundários (`twinx`) nos gráficos 1 e 3, anotações posicionais, formatação dos ticks com vírgula decimal e composição de subplots com proporções assimétricas no gráfico 2.

A escolha reflete a arquitetura das bibliotecas: o `seaborn` (WASKOM, 2021) é construído sobre o `matplotlib` (HUNTER, 2007) e o uso conjunto é o padrão recomendado.

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator
from scipy.stats import chi2_contingency, norm

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)

# Paleta institucional da Azul e tema unico para todas as figuras.
PALETA_AZUL = ['#00A0DF', '#2E5FA3', '#E8871E', '#C0392B']
AZ_ESC, AZ_CLA, CINZA, VERM, LARANJA = '#0A2A6B', '#00A0DF', '#9AA5B1', '#C0392B', '#E8871E'

sns.set_theme(style='whitegrid', context='notebook', palette=PALETA_AZUL,
              rc={'grid.linestyle': '--', 'grid.alpha': .25, 'axes.edgecolor': '#333'})
plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.dpi': 110, 'savefig.bbox': 'tight'})


def virgula(v, casas=1, sufixo=''):
    """Separador decimal em virgula, conforme a norma de escrita do projeto."""
    return f'{v:.{casas}f}'.replace('.', ',') + sufixo


def fmt(casas=0, sufixo=''):
    """Formatador de eixo independente do locale do ambiente."""
    return FuncFormatter(lambda v, _pos: virgula(v, casas, sufixo))


def rotulos(matriz, casas=1):
    'Anotacoes de heatmap com virgula decimal, ja que o fmt do seaborn usa ponto.'
    return np.array([['' if pd.isna(v) else virgula(v, casas) for v in linha]
                     for linha in np.asarray(matriz)])


# Onde gravar os PNGs. Os nomes sao os mesmos que documentacao.md referencia em
# assets/. Defina como None para apenas exibir, sem escrever em disco.
DIR_FIGURAS = 'figuras'


def mostrar(fig, nome):
    'Renderiza a figura como saida da celula e grava o PNG correspondente.'
    if DIR_FIGURAS:
        os.makedirs(DIR_FIGURAS, exist_ok=True)
        fig.savefig(os.path.join(DIR_FIGURAS, f'{nome}.png'), dpi=200, bbox_inches='tight')
    plt.show()


print('seaborn', sns.__version__, '| matplotlib', plt.matplotlib.__version__,
      '| pandas', pd.__version__)

### 1.1. Caminho das bases

Ajuste `CAMINHO_DADOS` conforme o ambiente. No Colab, descomente as duas linhas de montagem do Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# CAMINHO_DADOS = '/content/drive/MyDrive/SAFIRA/dados'

CAMINHO_DADOS = 'dados'

assert os.path.isdir(CAMINHO_DADOS), (
    f'Diretorio {CAMINHO_DADOS!r} nao encontrado. Aponte para a pasta com os cinco arquivos da Azul.')
sorted(os.path.basename(f) for f in glob.glob(os.path.join(CAMINHO_DADOS, '*.csv')))

---

## 2. Carga e integração das bases

O material recebido é composto por cinco arquivos, com duas naturezas distintas.

As três primeiras fontes são **transacionais** e compartilham a chave `RESPONDENT_ID` em relação **1:1**, com cobertura integral: toda resposta da pesquisa tem contrapartida operacional e de perfil. `NPS_01` a `NPS_04` trazem a pesquisa de satisfação, `INFORMACAO_VIAGEM` os dados operacionais do voo e `PERFIL_CLIENTE_01` e `_02` o perfil de relacionamento.

A quarta fonte, `DISTRIBUICAO_PAX_NORMALIZADO`, é **agregada** e não transacional. Ela informa a composição real do universo de passageiros por mês, faixa de atraso e canal de compra, e por isso serve de referência populacional para o diagnóstico de viés da seção 8.

In [ ]:
nps = pd.concat(
    [pd.read_csv(f, low_memory=False)
     for f in sorted(glob.glob(os.path.join(CAMINHO_DADOS, 'PROJETO_INTELI.NPS_0*.csv')))],
    ignore_index=True)
perfil = pd.concat(
    [pd.read_csv(f, low_memory=False)
     for f in sorted(glob.glob(os.path.join(CAMINHO_DADOS, 'PROJETO_INTELI.PERFIL_CLIENTE_0*.csv')))],
    ignore_index=True)
viagem = pd.read_csv(os.path.join(CAMINHO_DADOS, 'PROJETO_INTELI.INFORMACAO_VIAGEM.csv'),
                     low_memory=False)
dist = pd.read_csv(os.path.join(CAMINHO_DADOS, 'PROJETO_INTELI.DISTRIBUICAO_PAX_NORMALIZADO.csv'))

print(f'NPS      : {nps.shape[0]:>7} linhas x {nps.shape[1]:>3} colunas')
print(f'PERFIL   : {perfil.shape[0]:>7} linhas x {perfil.shape[1]:>3} colunas')
print(f'VIAGEM   : {viagem.shape[0]:>7} linhas x {viagem.shape[1]:>3} colunas')
print(f'DIST PAX : {dist.shape[0]:>7} linhas x {dist.shape[1]:>3} colunas  (base agregada)')

### 2.1. Checagem da chave antes da junção

Antes de juntar, confirmamos que a relação é de fato 1:1 e que a cobertura é integral. Se a interseção fosse menor que qualquer uma das bases, a junção interna descartaria registros silenciosamente.

In [ ]:
chave = 'RESPONDENT_ID'
ids_nps, ids_per, ids_via = set(nps[chave]), set(perfil[chave]), set(viagem[chave])

print(f'{chave} unicos em NPS   : {len(ids_nps)}  (duplicados: {len(nps) - len(ids_nps)})')
print(f'{chave} unicos em PERFIL: {len(ids_per)}  (duplicados: {len(perfil) - len(ids_per)})')
print(f'{chave} unicos em VIAGEM: {len(ids_via)}  (duplicados: {len(viagem) - len(ids_via)})')
print(f'\nIntersecao das tres bases: {len(ids_nps & ids_per & ids_via)}')
print(f'Cobertura de NPS pelas auxiliares: '
      f'{virgula(len(ids_nps & ids_per & ids_via) / len(ids_nps) * 100, 2, "%")}')

---

## 3. Filtros aplicados e tratamento de duplicidades

A verificação produziu um resultado atipicamente limpo, o que por si só informa sobre a maturidade do processo de extração da Azul. Cada filtro é registrado em log auditável.

| Filtro | Justificativa |
|---|---|
| F1. Linhas integralmente duplicadas | Duplicação de ingestão. Aplicado às três tabelas |
| F2. Chave duplicada com conteúdo divergente | Só prossegue se a divergência estiver em coluna previamente aprovada. Qualquer outra interrompe a execução |
| F3a. Cobertura da chave | Confere a correspondência antes da junção, porque um `inner join` descartaria respostas em silêncio |
| F3. Junção interna 1:1 | `validate="one_to_one"` do pandas, que interrompe se a cardinalidade declarada não se sustentar |
| F4. Colunas constantes | `VOO_INTERNACIONAL` assume `Domestic` em 100% dos registros, sem poder discriminativo |

Duas decisões merecem destaque, porque a alternativa cômoda seria silenciosa.

A primeira: **não usamos `drop_duplicates(subset='RESPONDENT_ID')` às cegas**. Manter a primeira ocorrência sem olhar o conteúdo descartaria dado válido sem aviso e tornaria o total de 484.915 registros uma contagem sem significado. Aqui, chave repetida com conteúdo divergente só é tolerada na coluna que a seção documenta como caso aprovado; qualquer outra levanta exceção.

A segunda: **a relação 1:1 é verificada, não presumida**. O parâmetro `validate="one_to_one"` do `pandas.merge` interrompe a execução se a chave não for única dos dois lados, em vez de multiplicar linhas silenciosamente.

In [ ]:
CHAVE = 'RESPONDENT_ID'

# A secao aprova exatamente uma divergencia: o registro 49088377 aparece duas
# vezes no NPS com TEMPO_VOO de 340 e 1.084 minutos. Qualquer outra coluna
# divergente entre copias da mesma chave interrompe a execucao.
DIVERGENCIA_APROVADA_NPS = ('TEMPO_VOO',)

log = []


def L(msg):
    log.append(msg)
    print(msg)


def duplicatas_divergentes(base, chave=CHAVE):
    'Mapeia chaves repetidas para as colunas em que as copias discordam.'
    repetidas = base[base.duplicated(chave, keep=False)]
    achados = {}
    for valor, grupo in repetidas.groupby(chave, observed=True):
        divergem = [c for c in grupo.columns
                    if c != chave and grupo[c].nunique(dropna=False) > 1]
        if divergem:
            achados[valor] = divergem
    return achados


def deduplicar(base, nome, divergencia_aprovada=(), chave=CHAVE):
    'Remove copias identicas e bloqueia divergencia de conteudo nao aprovada.'
    n0 = len(base)
    base = base.drop_duplicates()
    L(f'[F1:{nome}] Linhas integralmente duplicadas removidas: {n0 - len(base)}')

    achados = duplicatas_divergentes(base, chave)
    bloqueio = {k: v for k, v in achados.items() if set(v) - set(divergencia_aprovada)}
    if bloqueio:
        raise ValueError(
            f'{nome}: {len(bloqueio)} chave(s) repetida(s) com divergencia fora do '
            f'que foi aprovado: {list(bloqueio.items())[:5]}')

    if achados:
        n0 = len(base)
        base = base.drop_duplicates(subset=chave, keep='first')
        L(f'[F2:{nome}] {len(achados)} chave(s) com divergencia aprovada em '
          f'{sorted(set(divergencia_aprovada))}: {n0 - len(base)} linha(s) removida(s)')
        for valor, colunas in list(achados.items())[:10]:
            L(f'          {chave}={valor} divergia em {colunas}')
    return base


L(f'[0] Bruto: NPS={len(nps)} PERFIL={len(perfil)} VIAGEM={len(viagem)}')

nps_f = deduplicar(nps, 'nps', DIVERGENCIA_APROVADA_NPS)
perfil_f = deduplicar(perfil, 'perfil')
viagem_f = deduplicar(viagem, 'viagem')

# F3a: cobertura conferida ANTES do inner join, que descartaria em silencio.
ids_nps = set(nps_f[CHAVE])
for nome, aux in [('perfil', perfil_f), ('viagem', viagem_f)]:
    faltam = ids_nps - set(aux[CHAVE])
    L(f'[F3a] Cobertura do NPS em {nome}: '
      f'{(1 - len(faltam) / len(ids_nps)) * 100:.2f}% ({len(faltam)} sem correspondencia)')
    if faltam:
        raise ValueError(
            f'Cobertura incompleta em {nome}: {len(faltam)} chave(s) sem correspondencia. '
            'A juncao interna descartaria essas respostas sem aviso.')

# ID_GOLDENRECORD existe nas duas tabelas: so descarta a copia apos comprovar
# que ela coincide com a original.
par = nps_f[[CHAVE, 'ID_GOLDENRECORD']].merge(
    perfil_f[[CHAVE, 'ID_GOLDENRECORD']], on=CHAVE, how='inner', suffixes=('_esq', '_dir'))
esq, dir_ = par['ID_GOLDENRECORD_esq'], par['ID_GOLDENRECORD_dir']
# Nulo dos dois lados conta como equivalente: NaN != NaN em comparacao direta.
iguais = esq.eq(dir_) | (esq.isna() & dir_.isna())
if not iguais.all():
    raise ValueError(
        f'ID_GOLDENRECORD diverge entre NPS e PERFIL em {int((~iguais).sum())} '
        'registro(s); descarte da copia bloqueado.')
nulos_gold = int(esq.isna().sum())
if nulos_gold:
    L(f'[F3b] ID_GOLDENRECORD nulo em {nulos_gold} registro(s) nas duas tabelas: '
      'esses voos ficam fora do agrupamento por Cliente na modelagem')

# F3: validate interrompe se a cardinalidade 1:1 declarada nao se sustentar.
df = (nps_f
      .merge(perfil_f.drop(columns=['ID_GOLDENRECORD']), on=CHAVE,
             how='inner', validate='one_to_one')
      .merge(viagem_f, on=CHAVE, how='inner', validate='one_to_one'))
L(f'[F3] Apos juncao 1:1 validada: {len(df)} linhas x {df.shape[1]} colunas')

if len(df) != len(nps_f):
    raise ValueError(f'A juncao devolveu {len(df)} linhas contra {len(nps_f)} esperadas.')

const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=const)
L(f'[F4] Colunas constantes removidas: {const}')
L(f'[FINAL] Base analitica: {len(df)} linhas x {df.shape[1]} colunas')

Uma trava que nunca dispara não prova nada. A célula abaixo injeta uma duplicata divergente artificial e verifica que a integração recusa prosseguir. É teste, não transformação: nada é alterado na base real.

In [ ]:
teste = nps.head(2).copy()
teste.iloc[1, teste.columns.get_loc(CHAVE)] = teste.iloc[0][CHAVE]   # mesma chave
teste.iloc[1, teste.columns.get_loc('VOO_TIPO')] = 'VALOR_DIVERGENTE'

try:
    deduplicar(teste, 'teste_sintetico')
    print('FALHOU: a divergencia passou sem bloqueio')
except ValueError as erro:
    print('Trava funcionou. Mensagem recebida:')
    print(f'  {erro}')

### 3.1. Unidade de análise e dependência intracliente

Embora não haja duplicidade de chave, o volume de respostas é maior que o de Clientes: parte dos respondentes participou da pesquisa mais de uma vez. A decisão de manter ou colapsar esses registros define a unidade de análise de todo o projeto.

In [ ]:
df['DETRATOR'] = (df['NPS_PRINCIPAL'] == -100).astype(int)
df['CLASSE_NPS'] = df['NPS_PRINCIPAL'].map({100: 'Promotor', 0: 'Neutro', -100: 'Detrator'})
df['DATA_STD'] = pd.to_datetime(df['DATA_STD'])

# CANCELAMENTO_VOO costuma vir como booleano, mas se o CSV trouxer texto o
# astype(bool) marcaria a string 'False' como verdadeira. Normaliza uma vez so.
if not pd.api.types.is_bool_dtype(df['CANCELAMENTO_VOO']):
    df['CANCELAMENTO_VOO'] = (df['CANCELAMENTO_VOO'].astype(str).str.strip().str.lower()
                              .isin(['true', '1', 'sim', 's']))

n_resp = df['ID_GOLDENRECORD'].value_counts()
df['N_RESP_CLIENTE'] = df['ID_GOLDENRECORD'].map(n_resp)
recorrentes = (df['N_RESP_CLIENTE'] > 1).mean() * 100

print(f'Respostas             : {len(df)}')
print(f'Clientes distintos    : {df["ID_GOLDENRECORD"].nunique()}')
print(f'Respostas recorrentes : {virgula(recorrentes, 1, "%")}')
print(f'Maximo de respostas por Cliente: {int(n_resp.max())}')

# O mesmo Cliente muda de classificacao entre voos? E isso que justifica o voo
# como unidade de analise.
varia = df.groupby('ID_GOLDENRECORD')['DETRATOR'].nunique()
print(f'\nClientes que detrataram em um voo e nao em outro: {int((varia > 1).sum())}')

# A variavel resposta nao e distorcida pela recorrencia.
print('\nTaxa de detracao por numero de respostas do Cliente:')
faixa_resp = pd.cut(df['N_RESP_CLIENTE'], [0, 1, 3, 100],
                    labels=['1 resposta', '2 a 3', '4 ou mais'])
print(df.groupby(faixa_resp, observed=True)['DETRATOR'].mean().mul(100).round(2).to_string())

Os registros repetidos **não foram removidos**, por duas razões. Primeiro, a unidade de decisão da área de Customer Experience é o voo: o mesmo Cliente pode detratar em um voo com atraso e promover no seguinte, e colapsar por Cliente eliminaria exatamente a variação intraindividual que o modelo precisa aprender. Segundo, a recorrência não distorce a variável resposta, como mostra a tabela acima.

A célula seguinte quantifica a dependência que resta, e que impõe uma restrição à modelagem.

In [ ]:
# Entre Clientes com exatamente duas respostas, com que frequencia detratam nas duas?
dois = df[df['N_RESP_CLIENTE'] == 2]
por_cliente = dois.groupby('ID_GOLDENRECORD')['DETRATOR'].sum()
observado = (por_cliente == 2).mean() * 100
p = dois['DETRATOR'].mean()
esperado = p ** 2 * 100

print(f'Detrataram nas duas respostas, observado: {virgula(observado, 1, "%")}')
print(f'Esperado sob independencia (p^2)        : {virgula(esperado, 1, "%")}')
print(f'Razao observado / esperado              : {virgula(observado / esperado, 1, "x")}')

print('\nComposicao por tier, respondentes unicos vs. recorrentes:')
print(pd.crosstab(df['N_RESP_CLIENTE'] > 1, df['TIER_VIAGEM'], normalize='index')
      .mul(100).round(1).rename(index={False: 'Unico', True: 'Recorrente'}).to_string())

> **Restrição derivada para a fase de modelagem.** A partição entre treino e teste deverá ser **agrupada por `ID_GOLDENRECORD`** (`GroupKFold` ou `GroupShuffleSplit`). Uma partição aleatória simples permitiria que o mesmo Cliente figurasse em ambos os conjuntos, levando o modelo a memorizar padrões individuais e superestimando artificialmente as métricas de desempenho.

---

## 4. Classificação das colunas

A separação entre numéricas e categóricas é feita **programaticamente**, e não por lista fixa. O critério não é apenas o `dtype`: campos numéricos com no máximo duas categorias, como flags, são tratados como categóricos, porque a média deles não tem interpretação de escala.

In [ ]:
def classificar_colunas(base, limite_binario=2):
    """Separa as colunas em numericas e categoricas segundo dtype e cardinalidade."""
    num, cat, temporais = [], [], []
    for c in base.columns:
        s = base[c]
        if pd.api.types.is_datetime64_any_dtype(s):
            temporais.append(c)
        elif pd.api.types.is_bool_dtype(s):
            cat.append(c)
        elif pd.api.types.is_numeric_dtype(s):
            (cat if s.nunique(dropna=True) <= limite_binario else num).append(c)
        else:
            cat.append(c)
    return num, cat, temporais


num_auto, cat_auto, temp_auto = classificar_colunas(df)
print(f'Numericas  : {len(num_auto)}')
print(f'Categoricas: {len(cat_auto)}')
print(f'Temporais  : {len(temp_auto)}')

pd.DataFrame({
    'Tipo': ['Numérica'] * len(num_auto) + ['Categórica'] * len(cat_auto) + ['Temporal'] * len(temp_auto),
    'Coluna': num_auto + cat_auto + temp_auto,
}).groupby('Tipo')['Coluna'].apply(list).to_frame()

Da classificação automática são separados os conjuntos que efetivamente entram na análise descritiva. Ficam de fora os identificadores (`RESPONDENT_ID`, `ID_GOLDENRECORD`) e os campos `NPS_*` de etapa da jornada, que são coletados no mesmo instrumento que origina a variável resposta e, portanto, configurariam vazamento se usados como preditores, conforme estabelecido na seção 4.1.3.

In [ ]:
NUMERICAS = ['TEMPO_VOO', 'ESTATISTICA_ATRASOSAIDA', 'ATRASO_CHEGADA',
             'ANTECEDENCIA_CANCELAMENTO', 'QTDE_VIAGENS_12M', 'QTDE_VIAGENS_24M',
             'QTDE_VIAGENS_36M']

CATEGORICAS = ['CLASSE_NPS', 'VOO_TIPO', 'TIPO_ENTRETENIMENTO', 'CANAL_COMPRA',
               'SEGMENTO', 'TIER_VIAGEM', 'SUB_FIL_MOTIVOVIAGEM',
               'SUB_FIL_FREQUENCIAAZUL', 'CANCELAMENTO_VOO', 'SUB_ENTRETENIMENTO1',
               'SUB_ENTRETENIMENTO2', 'EQUIPAMENTO_TIPO', 'BASE_AIRPORTLEG',
               'ASSENTOS', 'VOO_NUMERO']

vazamento = [c for c in df.columns if c.startswith('NPS_') and c != 'NPS_PRINCIPAL']
print(f'Numericas analisadas  : {len(NUMERICAS)}')
print(f'Categoricas analisadas: {len(CATEGORICAS)}')
print(f'Excluidas por vazamento (pesquisa): {len(vazamento)} colunas {vazamento[:6]} ...')

---

## 5. Estatística descritiva

### 5.1. Variáveis numéricas

Além dos momentos usuais, a tabela traz o P95, a nulidade e a assimetria. A assimetria é o que decide o tratamento na fase de preparação: valores altos indicam que média e desvio padrão descrevem mal a distribuição.

In [ ]:
def tabela_numericas(base, cols):
    t = base[cols].describe(percentiles=[.25, .5, .75, .95]).T
    t['nulos'] = base[cols].isna().mean() * 100
    t['assimetria'] = base[cols].skew()
    t = t[['count', 'mean', '50%', 'std', 'min', 'max', '95%', 'nulos', 'assimetria']]
    t.columns = ['n', 'Média', 'Mediana', 'Desvio', 'Mín', 'Máx', 'P95', '% Nulo', 'Assimetria']
    return t.round(2)


tabela_numericas(df, NUMERICAS)

Todas as variáveis numéricas apresentam **forte assimetria positiva**, com mediana consistentemente inferior à média. Nos campos de atraso a mediana é zero, o que reflete uma operação majoritariamente pontual.

A consequência metodológica é dupla: transformações logarítmicas ou discretização em faixas serão preferíveis ao uso das variáveis em escala bruta, e métricas baseadas em média descrevem mal esses campos.

### 5.2. Variáveis categóricas

Para as categóricas, o que interessa é a cardinalidade, a concentração na moda e a nulidade. Cardinalidade muito alta inviabiliza codificação categórica convencional.

In [ ]:
def tabela_categoricas(base, cols):
    linhas = []
    for c in cols:
        vc = base[c].value_counts(dropna=True)
        linhas.append({'Variável': c,
                       'Categorias': int(base[c].nunique()),
                       'Moda': str(vc.index[0]),
                       'Freq. moda': int(vc.iloc[0]),
                       '% moda': round(vc.iloc[0] / len(base) * 100, 2),
                       '% Nulo': round(base[c].isna().mean() * 100, 2)})
    return pd.DataFrame(linhas).sort_values('Categorias').reset_index(drop=True)


tabela_categoricas(df, CATEGORICAS)

`BASE_AIRPORTLEG`, `ASSENTOS`, `EQUIPAMENTO_TIPO` e `VOO_NUMERO` apresentam cardinalidade muito elevada. Os três primeiros são campos compostos, cujo valor informacional será extraído por decomposição na seção 7. `VOO_NUMERO`, por identificar operações específicas, será descartado do conjunto de preditores.

### 5.3. Distribuição do alvo

`NPS_PRINCIPAL` assume três valores, correspondentes às classes definidas por Reichheld (2003) na formulação original do Net Promoter Score.

In [ ]:
dist_alvo = df['CLASSE_NPS'].value_counts(normalize=True).mul(100).round(2)
print(dist_alvo.to_string())
print(f'\nNPS bruto: {virgula(df["NPS_PRINCIPAL"].mean(), 1)}')
print(f'Desbalanceamento Detrator vs. nao-Detrator: '
      f'1 para {virgula((1 - df["DETRATOR"].mean()) / df["DETRATOR"].mean(), 1)}')

### 5.4. Força de associação das categóricas com o alvo

O coeficiente de correlação não se aplica a variáveis nominais. A medida adequada é o **V de Cramér** (CRAMÉR, 1946), derivado do qui-quadrado de contingência e normalizado para o intervalo de 0 a 1.

In [ ]:
def cramers_v(x, y):
    """V de Cramer entre duas variaveis nominais. 0 e independencia, 1 e associacao perfeita."""
    tab = pd.crosstab(x, y)
    if min(tab.shape) < 2:
        return np.nan
    chi2 = chi2_contingency(tab)[0]
    n = tab.to_numpy().sum()
    return float(np.sqrt(chi2 / (n * (min(tab.shape) - 1))))


CAT_ASSOCIACAO = ['CANCELAMENTO_VOO', 'SUB_FIL_FREQUENCIAAZUL', 'TIPO_ENTRETENIMENTO',
                  'VOO_TIPO', 'TIER_VIAGEM', 'SUB_FIL_MOTIVOVIAGEM', 'SEGMENTO',
                  'CANAL_COMPRA']

cramer = (pd.DataFrame([{'Variável': c, 'V de Cramér': round(cramers_v(df[c], df['DETRATOR']), 3)}
                        for c in CAT_ASSOCIACAO])
          .sort_values('V de Cramér', ascending=False).reset_index(drop=True))
cramer

Nenhuma variável categórica isolada apresenta associação forte com a detração. O resultado reforça o caráter **multivariado** do fenômeno e justifica a escolha de algoritmos capazes de capturar interações. `FAIXA_ATRASO` e `AEROPORTO_ORIGEM` entram nesta tabela apenas depois de construídas, na seção 7.

Vale destacar desde já o contraste que a seção 8 desenvolve: o canal de compra é o segundo maior eixo de viés amostral, mas praticamente não discrimina o alvo. Ele explica **quem responde**, não **quem detrata**.

---

## 6. Qualidade dos dados: inconsistências identificadas

Quatro achados, um por célula. Nenhum deles é erro de ingestão: todos exigem decisão metodológica explícita na fase de preparação.

### 6.1. Nulidade estrutural em `TIPO_ENTRETENIMENTO`

A hipótese é que a ausência não decorre de falha de coleta, mas da própria natureza do voo. A tabulação cruzada com `VOO_TIPO` testa isso.

In [ ]:
nulos_ent = df['TIPO_ENTRETENIMENTO'].isna()
print(f'Nulos em TIPO_ENTRETENIMENTO: {int(nulos_ent.sum())} '
      f'({virgula(nulos_ent.mean() * 100, 2, "%")})')
print('\nTabulacao cruzada com VOO_TIPO:')
print(pd.crosstab(df['VOO_TIPO'], nulos_ent).rename(
    columns={False: 'Preenchido', True: 'Nulo'}).to_string())

print('\nANTECEDENCIA_CANCELAMENTO condicionada a CANCELAMENTO_VOO:')
print(pd.crosstab(df['CANCELAMENTO_VOO'],
                  df['ANTECEDENCIA_CANCELAMENTO'].isna()).rename(
    columns={False: 'Preenchido', True: 'Nulo'}).to_string())

A correspondência é exata: os registros nulos são precisamente os voos de conexão. Como uma conexão envolve mais de uma aeronave, não existe um único sistema de entretenimento associado ao trecho, e a imputação seria conceitualmente incorreta. O tratamento adequado é a criação de uma categoria explícita, `Não aplicável (conexão)`.

O mesmo raciocínio vale para `ANTECEDENCIA_CANCELAMENTO`: o campo está preenchido em 100% dos voos cancelados e nulo em 100% dos não cancelados, sendo **condicionado** a `CANCELAMENTO_VOO` e não faltante.

### 6.2. Divergência entre os campos de atraso

`ESTATISTICA_ATRASOSAIDA` e `ATRASO_CHEGADA` deveriam medir o mesmo evento operacional em dois momentos. A correlação entre eles testa se de fato medem.

In [ ]:
rho = df[['ESTATISTICA_ATRASOSAIDA', 'ATRASO_CHEGADA']].corr(method='spearman').iloc[0, 1]
print(f'Correlacao de Spearman entre os campos de atraso: {virgula(rho, 3)}')

incoerente_a = ((df['ESTATISTICA_ATRASOSAIDA'] < 15) & (df['ATRASO_CHEGADA'] > 60)).sum()
incoerente_b = ((df['ESTATISTICA_ATRASOSAIDA'] > 60) & (df['ATRASO_CHEGADA'] == 0)).sum()
print(f'Partida pontual e chegada com mais de 60 min de atraso: {int(incoerente_a)}')
print(f'Partida atrasada e chegada registrada como pontual    : {int(incoerente_b)}')

print(f'\nATRASO_CHEGADA igual a zero: '
      f'{virgula((df["ATRASO_CHEGADA"] == 0).mean() * 100, 1, "%")}')
print(f'Maximo de ATRASO_CHEGADA   : {int(df["ATRASO_CHEGADA"].max())} min')
print('\nMedia de ATRASO_CHEGADA por status do voo:')
print(df.groupby('CANCELAMENTO_VOO')['ATRASO_CHEGADA'].mean().round(1).to_string())

A hipótese de trabalho é que o campo **agrega semânticas distintas**: chegada antecipada codificada como zero, e tempo até a reacomodação nos casos de cancelamento. A observação é consistente com o apontamento da própria equipe da Azul quanto à necessidade de revisão dos campos usados em situações de atraso.

**A validação da regra de cálculo de `ATRASO_CHEGADA` foi encaminhada ao ponto focal da Azul como pendência.**

### 6.3. Inconsistência entre frequência declarada e observada

`SUB_FIL_FREQUENCIAAZUL` é autodeclarada pelo respondente. `QTDE_VIAGENS_36M` vem do registro operacional. Confrontar as duas mede o erro do instrumento.

In [ ]:
primeira = df[df['SUB_FIL_FREQUENCIAAZUL'].astype(str).str.contains('primeira', case=False, na=False)]
tiers_altos = ['Diamante', 'Safira', 'Topazio']

print(f'Declararam "Esta foi a primeira vez": {len(primeira)}')
if len(primeira):
    print(f'  com mais de uma viagem em 36 meses: '
          f'{virgula((primeira["QTDE_VIAGENS_36M"] > 1).mean() * 100, 1, "%")}')
    print(f'  pertencentes a Diamante, Safira ou Topazio: '
          f'{virgula(primeira["TIER_VIAGEM"].isin(tiers_altos).mean() * 100, 1, "%")}')

print('\nMediana de viagens em 36 meses por frequencia declarada:')
print(df.groupby('SUB_FIL_FREQUENCIAAZUL')['QTDE_VIAGENS_36M'].median().to_string())

A divergência pode refletir ambiguidade na formulação da pergunta, referindo-se à primeira vez naquela rota e não na companhia, ou erro de recordação. Independentemente da causa, o campo apresenta **erro de medida substancial** e, por ser coletado no mesmo instrumento que origina a variável resposta, acumula risco de vazamento.

**Recomenda-se seu descarte em favor de `QTDE_VIAGENS_12M`**, que mensura o mesmo construto a partir de registro operacional.

### 6.4. Outliers em `TEMPO_VOO`

O valor máximo é implausível para operação doméstica. A pergunta é se é erro ou se o campo mede outra coisa.

In [ ]:
print(f'Maximo de TEMPO_VOO: {int(df["TEMPO_VOO"].max())} min '
      f'({virgula(df["TEMPO_VOO"].max() / 60, 1)} horas)')
print('\nDistribuicao por tipo de voo:')
print(df.groupby('VOO_TIPO')['TEMPO_VOO']
      .agg(n='size', mediana='median', p99=lambda s: s.quantile(.99), maximo='max')
      .round(1).to_string())
print(f'\nVoos diretos acima de 600 min: '
      f'{int(((df["VOO_TIPO"] == "Direto") & (df["TEMPO_VOO"] > 600)).sum())}')

A distribuição é aceitável em voos Diretos e apresenta cauda extensa em Conexões. Como `TEMPO_VOO` mede a **viagem completa** e não o tempo em voo, valores elevados em conexões refletem esperas prolongadas: informação legítima e potencialmente preditiva.

O tratamento será por **winsorização no P99 dentro de cada tipo de voo**, e não por exclusão de registros.

---

## 7. Variáveis derivadas

Seis variáveis são construídas nesta etapa. `PESO_POP` é calculada apenas na seção 8, por depender da base populacional.

| Variável | Origem | Justificativa |
|---|---|---|
| `N_TRECHOS` | Contagem de separadores em `BASE_AIRPORTLEG` | Complexidade do itinerário |
| `AEROPORTO_ORIGEM` e `AEROPORTO_DESTINO` | Decomposição de `BASE_AIRPORTLEG` | Reduz a cardinalidade de milhares de itinerários para poucas centenas de aeroportos |
| `FAIXA_ATRASO` | Discretização de `ESTATISTICA_ATRASOSAIDA` | Compatibiliza com a taxonomia usada pela Azul e com a base populacional |
| `MES_ANO` e `TRIMESTRE` | Extração de `DATA_STD` | Capturam o efeito sazonal e o efeito de período |

In [ ]:
ORD_ATRASO = ['a. Sem Atraso', 'b. 15m - 60m', 'c. 61m - 120m', 'd. >120m']
LAB_ATRASO = ['Sem atraso\n(<15 min)', '15 a 60 min', '61 a 120 min', 'Acima de 120 min']

df['N_TRECHOS'] = df['BASE_AIRPORTLEG'].str.count('/')          # trechos = aeroportos - 1
df['AEROPORTO_ORIGEM'] = df['BASE_AIRPORTLEG'].str.split('/').str[0]
df['AEROPORTO_DESTINO'] = df['BASE_AIRPORTLEG'].str.split('/').str[-1]

banda = np.select([df['ESTATISTICA_ATRASOSAIDA'] < 15,
                   df['ESTATISTICA_ATRASOSAIDA'] <= 60,
                   df['ESTATISTICA_ATRASOSAIDA'] <= 120],
                  ORD_ATRASO[:3], default=ORD_ATRASO[3])
df['FAIXA_ATRASO'] = pd.Categorical(banda, categories=ORD_ATRASO, ordered=True)

df['MES_ANO'] = df['DATA_STD'].dt.to_period('M').dt.to_timestamp()
df['TRIMESTRE'] = df['DATA_STD'].dt.to_period('Q').astype(str)

print('Cardinalidade antes e depois da decomposicao:')
print(f'  BASE_AIRPORTLEG : {df["BASE_AIRPORTLEG"].nunique()} itinerarios')
print(f'  AEROPORTO_ORIGEM: {df["AEROPORTO_ORIGEM"].nunique()} aeroportos')
print(f'\nDistribuicao de FAIXA_ATRASO:')
print(df['FAIXA_ATRASO'].value_counts(normalize=True).mul(100).round(2).to_string())

### 7.1. Relação de `N_TRECHOS` com o alvo

A hipótese é que a complexidade do itinerário seja, isoladamente, um fator de risco.

In [ ]:
print('Taxa de detratores por numero de trechos:')
print(df.groupby('N_TRECHOS')['DETRATOR'].agg(n='size', taxa=lambda s: s.mean() * 100)
      .round(2).to_string())
print(f'\nV de Cramer de FAIXA_ATRASO com o alvo    : '
      f'{virgula(cramers_v(df["FAIXA_ATRASO"], df["DETRATOR"]), 3)}')
print(f'V de Cramer de AEROPORTO_ORIGEM com o alvo: '
      f'{virgula(cramers_v(df["AEROPORTO_ORIGEM"], df["DETRATOR"]), 3)}')

A relação é **monotônica** e a variável está disponível no momento da predição, sem risco de vazamento. `FAIXA_ATRASO` passa a ser a categórica de maior associação com o alvo em toda a base.

### 7.2. Derivação descartada: `ASSENTOS` como tamanho de grupo

Este registro é de rigor metodológico, não erro a esconder. O campo `ASSENTOS` foi inicialmente interpretado como indicador do tamanho do grupo viajante. A verificação cruzada refutou a hipótese, e o teste está reproduzido abaixo.

In [ ]:
df['N_ASSENTOS_AUDIT'] = df['ASSENTOS'].str.count('/').add(1)

cruzada = pd.crosstab(df['N_ASSENTOS_AUDIT'], df['N_TRECHOS'])
print('Tabulacao cruzada N_ASSENTOS_AUDIT x N_TRECHOS:')
print(cruzada.to_string())

par = df[['N_ASSENTOS_AUDIT', 'N_TRECHOS']].dropna()
rho_assentos = par.corr(method='spearman').iloc[0, 1]
diagonal = (par['N_ASSENTOS_AUDIT'] == par['N_TRECHOS']).mean() * 100
print(f'\nCorrelacao de Spearman: {virgula(rho_assentos, 3)}')
print(f'Casos na diagonal (contagens iguais): {virgula(diagonal, 1, "%")}')

A correspondência é quase perfeita. O registro `20A/17A/28A`, associado ao itinerário `FOR/UDI/CNF/POA`, corresponde a **três assentos do mesmo passageiro em três trechos consecutivos**, e não a três passageiros.

A variável foi mantida apenas como campo de auditoria e **excluída do conjunto de preditores por redundância** com `N_TRECHOS`. Nenhum campo da base permite identificar viagens em grupo, limitação registrada como pedido de dado adicional ao parceiro.

---

## 8. Representatividade e viés amostral

Esta é a contribuição analítica central da exploração. A pesquisa de NPS da Azul é enviada a aproximadamente metade dos Clientes domésticos, com taxa de resposta inferior a 10%. A amostra é, portanto, **autosselecionada**.

A literatura de survey demonstra que baixas taxas de resposta não geram viés por si mesmas, mas o produzem quando a propensão a responder se correlaciona com a variável de interesse (GROVES; PEYTCHEVA, 2008). A base `DISTRIBUICAO_PAX_NORMALIZADO` permite testar exatamente essa condição.

### 8.1. Viés de resposta associado ao atraso

In [ ]:
pop_atraso = dist.groupby('DELAY_DEPARTURE_RANGE')['PERC_PAX'].sum()
pop_atraso = (pop_atraso / pop_atraso.sum() * 100).reindex(ORD_ATRASO)
amo_atraso = df['FAIXA_ATRASO'].value_counts(normalize=True).mul(100).reindex(ORD_ATRASO)

comp = pd.DataFrame({
    '% população PAX': pop_atraso.round(2),
    '% amostra': amo_atraso.round(2),
    'Razão': (amo_atraso / pop_atraso).round(2),
    'Taxa de detratores': df.groupby('FAIXA_ATRASO', observed=True)['DETRATOR']
                            .mean().mul(100).reindex(ORD_ATRASO).round(2),
})
comp

Passageiros que sofreram atraso superior a 120 minutos respondem à pesquisa muito acima do que sua participação na operação justificaria, e detratam a uma taxa várias vezes superior à dos voos pontuais.

Este é o cenário mais crítico de viés amostral: **a propensão a responder está correlacionada com a variável-alvo**.

### 8.2. Viés de canal de compra

In [ ]:
pop_canal = dist.groupby('CANAL_COMPRA')['PERC_PAX'].sum()
pop_canal = (pop_canal / pop_canal.sum() * 100)
amo_canal = df['CANAL_COMPRA'].value_counts(normalize=True).mul(100)

canal = pd.DataFrame({'% população PAX': pop_canal.round(2),
                      '% amostra': amo_canal.round(2)})
canal['Razão'] = (canal['% amostra'] / canal['% população PAX']).round(2)
canal['Taxa de detratores'] = (df.groupby('CANAL_COMPRA')['DETRATOR']
                               .mean().mul(100).round(2))
canal.sort_values('% população PAX', ascending=False)

Clientes que compram via agência estão sub-representados na amostra, enquanto os canais digitais aparecem sobre-representados. O padrão confirma a hipótese levantada pela equipe da Azul de que o contato indireto com a companhia reduz a taxa de resposta.

Note, porém, que a coluna de detração é praticamente plana entre canais. Este é o contraste que a seção 5.4 antecipou: o canal explica **quem responde**, não **quem detrata**.

### 8.3. Pós-estratificação

A pós-estratificação corrige desvios de composição atribuindo a cada observação um peso proporcional à razão entre sua frequência na população e na amostra (VALLIANT, 1993). A estratificação é feita por **mês, faixa de atraso e canal de compra**.

A cobertura integral dessa chave **é condição de parada, não um aviso**. Um estrato da amostra sem contrapartida populacional receberia peso indefinido; preenchê-lo com zero removeria aqueles respondentes do estimador ponderado em silêncio e deslocaria a taxa de detração sem que nada indicasse o problema. A célula abaixo interrompe a execução e exibe as chaves ausentes se isso ocorrer.

In [ ]:
dist['MES_ANO'] = pd.to_datetime(dist['MES'])

pop = (dist.groupby(['MES_ANO', 'DELAY_DEPARTURE_RANGE', 'CANAL_COMPRA'])['PERC_PAX']
       .sum().rename('p_pop').reset_index())
pop['p_pop'] = pop.groupby('MES_ANO')['p_pop'].transform(lambda x: x / x.sum())

amo = (df.groupby(['MES_ANO', 'FAIXA_ATRASO', 'CANAL_COMPRA'], observed=True)
       .size().rename('n').reset_index())
amo['p_amo'] = amo.groupby('MES_ANO')['n'].transform(lambda x: x / x.sum())

m = amo.merge(pop, left_on=['MES_ANO', 'FAIXA_ATRASO', 'CANAL_COMPRA'],
              right_on=['MES_ANO', 'DELAY_DEPARTURE_RANGE', 'CANAL_COMPRA'], how='left')
m['PESO_POP'] = m['p_pop'] / m['p_amo']

ESTRATOS = ['MES_ANO', 'FAIXA_ATRASO', 'CANAL_COMPRA']

# Estrato sem contrapartida populacional interrompe: peso zero seria exclusao
# silenciosa do estimador ponderado.
sem_peso = m[m['PESO_POP'].isna() | (m['PESO_POP'] <= 0)]
if len(sem_peso):
    raise ValueError(
        f'Pos-estratificacao incompleta: {len(sem_peso)} estrato(s) sem peso valido, '
        f'cobrindo {int(sem_peso["n"].sum())} resposta(s). Chaves ausentes: '
        + sem_peso[ESTRATOS + ['n']].head(10).to_string(index=False))

df = df.merge(m[ESTRATOS + ['PESO_POP']], on=ESTRATOS, how='left')

orfaos = df['PESO_POP'].isna()
if orfaos.any():
    raise ValueError(
        f'{int(orfaos.sum())} resposta(s) ficaram sem PESO_POP apos a juncao: Chaves ausentes: '
        + df.loc[orfaos, ESTRATOS].drop_duplicates().head(10).to_string(index=False))

print(f'Estratos com peso valido: {len(m)}')
print(f'Cobertura da chave de pos-estratificacao: '
      f'{virgula((df["PESO_POP"] > 0).mean() * 100, 2, "%")}')

#### Diagnóstico dos pesos

Pesos válidos não bastam: pesos muito desiguais inflam a variância das métricas ponderadas mesmo com cobertura completa. O **tamanho amostral efetivo de Kish**, dado por (soma dos pesos)² dividida pela soma dos pesos ao quadrado, responde a quantas observações sem peso esta amostra ponderada equivale.

In [ ]:
w = df['PESO_POP'].to_numpy(dtype=float)
n_efetivo = w.sum() ** 2 / np.square(w).sum()

diagnostico = pd.Series({
    'n': len(w),
    'Mínimo': w.min(),
    'P1': np.quantile(w, .01),
    'Mediana': np.median(w),
    'P99': np.quantile(w, .99),
    'Máximo': w.max(),
    'Razão P99/P1': np.quantile(w, .99) / np.quantile(w, .01),
    'n efetivo (Kish)': n_efetivo,
    'Perda de eficiência (%)': (1 - n_efetivo / len(w)) * 100,
})
diagnostico.round(3)

#### Métricas ponderadas com intervalo de confiança

Sem intervalo não é possível afirmar que a diferença entre bruto e ponderado é deslocamento real de composição, e não ruído. O intervalo da média ponderada usa o estimador de Hájek, com variância igual à soma de w²(y − média)² dividida pelo quadrado da soma dos pesos. Ele trata os pesos como fixos e ignora o desenho estratificado, portanto é aproximação, e tende a ser conservador.

In [ ]:
def media_ponderada_ic(y, w, confianca=.95):
    'Media ponderada de Hajek com intervalo por linearizacao.'
    y, w = np.asarray(y, float), np.asarray(w, float)
    soma = w.sum()
    media = np.sum(w * y) / soma
    erro = np.sqrt(np.sum(np.square(w) * np.square(y - media)) / soma ** 2)
    z = norm.ppf(.5 + confianca / 2)
    return media, media - z * erro, media + z * erro


def media_simples_ic(y, confianca=.95):
    'Media simples com intervalo, para comparacao.'
    y = np.asarray(y, float)
    media, erro = y.mean(), y.std(ddof=1) / np.sqrt(len(y))
    z = norm.ppf(.5 + confianca / 2)
    return media, media - z * erro, media + z * erro


linhas = []
for coluna, rotulo, escala in [('DETRATOR', 'Taxa de detratores (%)', 100),
                               ('NPS_PRINCIPAL', 'NPS médio', 1)]:
    bruto = media_simples_ic(df[coluna])
    pond = media_ponderada_ic(df[coluna], df['PESO_POP'])
    linhas.append({
        'Métrica': rotulo,
        'Bruto': bruto[0] * escala,
        'IC 95% bruto': f'[{virgula(bruto[1] * escala, 2)}; {virgula(bruto[2] * escala, 2)}]',
        'Ponderado': pond[0] * escala,
        'IC 95% ponderado': f'[{virgula(pond[1] * escala, 2)}; {virgula(pond[2] * escala, 2)}]',
    })

resumo = pd.DataFrame(linhas).set_index('Métrica')
resumo.round(2)

A amostra bruta **superestima a detração**. O resultado ganha credibilidade por um teste externo: o NPS pós-estratificado situa-se dentro da faixa que a Azul declara como seu patamar corrente, enquanto o valor bruto fica abaixo dela. A ponderação reconcilia a amostra com a métrica oficial da companhia.

Conforme decisão da equipe, o viés é **diagnosticado nesta fase e sua incorporação será decidida na etapa de modelagem**, quando serão avaliadas as alternativas de uso dos pesos no treinamento, na avaliação, ou apenas na comunicação ao negócio.

### 8.4. Efeito de período

A série trimestral testa se a detração varia além do que a operação explica. As duas séries são plotadas juntas justamente para permitir esse contraste: se a detração subisse na mesma proporção dos atrasos, seria composição operacional.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

t = (df.groupby('TRIMESTRE', observed=True)
       .agg(Detratores=('DETRATOR', 'mean'),
            Atrasos=('FAIXA_ATRASO', lambda s: (s != ORD_ATRASO[0]).mean()))
       .mul(100).reset_index())
longo = t.melt('TRIMESTRE', var_name='Série', value_name='pct')

# seaborn: duas series no mesmo eixo, cor e marcador pela coluna 'Serie'.
sns.lineplot(data=longo, x='TRIMESTRE', y='pct', hue='Série', style='Série',
             markers=['o', 's'], dashes=False, lw=2.4, markersize=7,
             palette=[VERM, AZ_CLA], ax=ax)

# matplotlib: destaque e anotacao do trimestre de pico.
pico = t.loc[t['Detratores'].idxmax()]
i = int(t.index[t['TRIMESTRE'] == pico['TRIMESTRE']][0])
ax.axvspan(i - .5, i + .5, color=LARANJA, alpha=.13, zorder=0)

# A anotacao fica acima das duas series, nunca em coordenada fixa.
topo = float(t[['Detratores', 'Atrasos']].to_numpy().max())
piso = float(t[['Detratores', 'Atrasos']].to_numpy().min())
ax.annotate(f'Pico de {pico["TRIMESTRE"]}\n{virgula(pico["Detratores"], 1, "%")} de detratores',
            (i, pico['Detratores']), xytext=(i + 1.2, topo + 3.5),
            fontsize=9.5, fontweight='bold', color=VERM,
            arrowprops=dict(arrowstyle='->', color=VERM, lw=1.4))

ax.set(xlabel='', ylabel='Percentual', ylim=(piso - 4, topo + 9))
ax.yaxis.set_major_formatter(fmt(0, '%'))
ax.legend(frameon=False, loc='upper left', title=None)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('Evolução trimestral da detração e da incidência de atrasos', pad=12)
ax.text(0, -.30, 'A detração do trimestre de pico sobe acima do que a variação de atrasos '
        'explica, indicando efeito de período.', transform=ax.transAxes,
        fontsize=8.5, color='#555')
mostrar(fig, 'g2_serie_temporal')

A análise condicional confirma que o fenômeno **não é explicado pela composição operacional**: a detração sobe dentro de todas as faixas de atraso, inclusive entre voos pontuais. O período coincide com o contexto que antecedeu a reestruturação financeira concluída pela companhia, o que sugere componente reputacional externo à operação do voo.

Duas implicações: a validação do modelo deverá adotar **partição temporal**, para não vazar informação de conjuntura entre treino e teste, e a variável temporal deve ser tratada como **covariável de contexto**, não como preditor estável.

### 8.5. Piso irredutível de detração

Isolando o cenário operacionalmente ideal, mede-se quanto da detração **não é operacionalmente evitável**. É o que delimita o teto de desempenho realista do projeto.

In [ ]:
ideal = df[(df['VOO_TIPO'] == 'Direto') & (~df['CANCELAMENTO_VOO'].astype(bool)) &
           (df['ESTATISTICA_ATRASOSAIDA'] < 15) & (df['ATRASO_CHEGADA'] < 15)]

print(f'Registros em condicao operacional ideal: {len(ideal)} '
      f'({virgula(len(ideal) / len(df) * 100, 1, "%")} da base)')
print(f'NPS medio          : {virgula(ideal["NPS_PRINCIPAL"].mean(), 1)}')
print(f'Taxa de detratores : {virgula(ideal["DETRATOR"].mean() * 100, 1, "%")}')
print('\nTaxa de detratores por tier, na condicao ideal:')
print(ideal.groupby('TIER_VIAGEM')['DETRATOR'].mean().mul(100).round(1)
      .sort_values(ascending=False).to_string())

Mesmo na ausência completa de falha operacional, uma parcela relevante dos Clientes avalia a experiência entre 0 e 6. Para a modelagem, isso indica que preditores puramente operacionais terão limite de poder discriminativo.

---

## 9. Análise das relações entre variáveis

Seis figuras, cada uma respondendo a uma pergunta específica do escopo definido com a Azul. Todas são construídas na própria célula e renderizadas como saída de execução.

### Gráfico 1. Atraso na saída: efeito sobre a detração e viés de resposta

*Tipo:* barras com eixo secundário. *Variáveis:* `FAIXA_ATRASO` (categórica derivada), taxa de detratores (numérica) e razão de representatividade (numérica).

A figura sobrepõe deliberadamente dois fenômenos que a literatura de pesquisa costuma tratar em separado. As barras (`seaborn.barplot`) mostram o efeito dose-resposta; a linha no eixo secundário (`matplotlib.twinx` mais `seaborn.lineplot`) mostra a sobre-representação da mesma faixa na pesquisa.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

base = (df.groupby('FAIXA_ATRASO', observed=True)['DETRATOR']
          .mean().mul(100).reindex(ORD_ATRASO).reset_index())

# seaborn: camada de dados, ordem das categorias e paleta institucional.
sns.barplot(data=base, x='FAIXA_ATRASO', y='DETRATOR', order=ORD_ATRASO,
            hue='FAIXA_ATRASO', palette=PALETA_AZUL, legend=False, width=.55, ax=ax)
for i, v in enumerate(base['DETRATOR']):
    ax.text(i, v + 1.8, virgula(v, 1, '%'), ha='center', fontweight='bold', fontsize=11)

razao = (amo_atraso / pop_atraso).to_numpy()

ax.set(xlabel='', ylabel='Taxa de detratores', ylim=(0, base['DETRATOR'].max() * 1.22))
ax.set_xticks(range(4))
ax.set_xticklabels(LAB_ATRASO)
ax.yaxis.set_major_formatter(fmt(0, '%'))

# matplotlib: eixo secundario, escala e unidade diferentes da esquerda.
ax2 = ax.twinx()
ax2.grid(False)
sns.lineplot(x=range(4), y=razao, marker='o', color=AZ_ESC, lw=2,
             markersize=8, linestyle='--', ax=ax2)
ax2.axhline(1, color=CINZA, lw=1.2, ls=':')
ax2.set(ylabel='Amostra dividida pela população de PAX',
        ylim=(0, max(2.2, float(razao.max()) * 1.45)))
ax2.yaxis.set_major_formatter(fmt(1))
ax2.yaxis.label.set_color(AZ_ESC)
ax2.tick_params(axis='y', colors=AZ_ESC)
for i, r in enumerate(razao):
    # Rotulo acima ou abaixo do marcador conforme a altura, com fundo branco,
    # para nunca colidir com o rotulo da barra.
    desloc = (0, 12) if r < razao.max() * .6 else (0, -18)
    ax2.annotate(virgula(r, 2, 'x'), (i, r), textcoords='offset points', xytext=desloc,
                 ha='center', color=AZ_ESC, fontsize=9, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=.15', fc='white', ec='none', alpha=.75))

ax.set_title('Atraso na saída: efeito sobre a detração e viés de resposta', pad=14)
ax.text(0, -.20, 'Barras: taxa de detratores por faixa. Linha: razão entre a participação '
        'na pesquisa e na população de passageiros.', transform=ax.transAxes,
        fontsize=8.5, color='#555')
mostrar(fig, 'g1_atraso_dose_resposta')

**Leitura.** As barras evidenciam uma relação **dose-resposta monotônica** e de magnitude expressiva. A linha revela que essas mesmas faixas são as mais sobre-representadas na pesquisa: a linha pontilhada em 1,00 marca a representatividade perfeita, e todas as faixas com atraso estão acima dela.

A leitura conjunta é o principal insight desta exploração. O atraso é simultaneamente o maior driver de insatisfação e o maior fator de distorção amostral. Qualquer modelo treinado sobre a amostra bruta herdará essa distorção, e qualquer indicador de detração calculado sem ponderação estará inflado.

### Gráfico 2. Limiar de atraso: curva de risco e impacto marginal

*Tipo:* série de linha com painel de variação marginal. *Variáveis:* `ESTATISTICA_ATRASOSAIDA` discretizada em treze faixas e taxa de detratores.

Responde diretamente à pergunta 5 do escopo definido pela Azul: existe um limiar de atraso a partir do qual o risco de detração aumenta significativamente? A composição usa `gridspec` do matplotlib com proporção 2,2 para 1, porque o painel inferior é leitura de apoio e não deve competir visualmente com a curva.

In [ ]:
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True,
                              gridspec_kw={'height_ratios': [2.2, 1]})

BINS = [-1, 0, 5, 10, 15, 20, 30, 45, 60, 90, 120, 180, 240, 10000]
LAB = ['0', '1-5', '6-10', '11-15', '16-20', '21-30', '31-45', '46-60',
       '61-90', '91-120', '121-180', '181-240', '>240']

faixa = pd.cut(df['ESTATISTICA_ATRASOSAIDA'], BINS, labels=LAB)
r = df.groupby(faixa, observed=False)['DETRATOR'].mean().mul(100).reset_index()
r.columns = ['faixa', 'taxa']
r = r.dropna(subset=['taxa']).reset_index(drop=True)
r['marginal'] = r['taxa'].diff()

# Painel superior: curva de risco.
sns.lineplot(data=r, x='faixa', y='taxa', marker='o', color=VERM, lw=2.6, markersize=7, ax=ax)
ax.fill_between(range(len(r)), r['taxa'], color=VERM, alpha=.10)

media = df['DETRATOR'].mean() * 100
ax.axhline(media, color=CINZA, ls=':', lw=1.5)
ax.text(.3, media + 2.5, f'média geral {virgula(media, 1, "%")}', color='#666',
        fontsize=8.5, ha='left')

rot = list(r['faixa'].astype(str))
if '21-30' in rot:
    i = rot.index('21-30')
    ax.axvspan(i - 1.5, i + .5, color=LARANJA, alpha=.16, zorder=0)
    ax.annotate('Ponto de inflexão\n20 a 30 min', (i, float(r.loc[i, 'taxa'])),
                xytext=(i + 2, max(media - 13, 4)), fontsize=10, fontweight='bold',
                color=LARANJA, arrowprops=dict(arrowstyle='->', color=LARANJA, lw=1.6))

ax.set(xlabel='', ylabel='Taxa de detratores')
ax.yaxis.set_major_formatter(fmt(0, '%'))
ax.set_title('Limiar de atraso: curva de risco e impacto marginal', pad=12)

# Painel inferior: variacao em pontos percentuais entre faixas consecutivas.
r['acelera'] = np.where(r['marginal'] >= 4, 'Acima de 4 p.p.', 'Até 4 p.p.')
sns.barplot(data=r, x='faixa', y='marginal', hue='acelera',
            palette={'Até 4 p.p.': AZ_CLA, 'Acima de 4 p.p.': LARANJA},
            dodge=False, width=.6, ax=ax2)
ax2.axhline(4, color=LARANJA, ls='--', lw=1.2)
if ax2.get_legend() is not None:
    ax2.get_legend().remove()
ax2.set(ylabel='Variação em p.p. vs.\nfaixa anterior', xlabel='Atraso na saída (minutos)')
ax2.yaxis.set_major_formatter(fmt(0))
ax2.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.text(0, -.85, 'Até 15 min o custo marginal é estável. A partir de 20 min ele dobra '
         'e segue acelerando.', transform=ax2.transAxes, fontsize=8.5, color='#555')
mostrar(fig, 'g7_limiar_atraso')

**Leitura.** A resposta é afirmativa e localizável. O painel inferior mostra que **até 15 minutos o custo marginal do atraso é estável**, na ordem de 2 p.p. por faixa. A partir de 20 minutos esse custo dobra e segue acelerando até a faixa de 61 a 90 minutos, quando atinge o máximo. Acima de 180 minutos o incremento desacelera, por efeito de saturação, já que a taxa se aproxima de 80%.

A leitura operacional é que **a janela de 20 a 30 minutos é o ponto de maior retorno para a atuação preventiva**. É onde a curva muda de regime e onde a intervenção ainda alcança um contingente grande de Clientes. Recomenda-se que este intervalo seja considerado na definição do *threshold* de acionamento do modelo.

### Gráfico 3. Cancelamento: efeito da antecedência do aviso

*Tipo:* barras com eixo secundário. *Variáveis:* `ANTECEDENCIA_CANCELAMENTO` discretizada, taxa de detratores e NPS médio. Recorte: apenas voos cancelados.

O desenho mantém o evento negativo constante, que é o cancelamento, e varia apenas a comunicação. É o que permite isolar o efeito do aviso do efeito do próprio cancelamento.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

c = df[df['CANCELAMENTO_VOO'].astype(bool)].copy()
c['FX'] = pd.cut(c['ANTECEDENCIA_CANCELAMENTO'], [-1, 0, 1, 3, 7, 15, 30, 60, 1000],
                 labels=['Mesmo dia', '1 dia', '2 a 3 d', '4 a 7 d', '8 a 15 d',
                         '16 a 30 d', '31 a 60 d', 'Mais de 60 d'])
s = (c.groupby('FX', observed=True)
       .agg(taxa=('DETRATOR', 'mean'), nps=('NPS_PRINCIPAL', 'mean')).reset_index())
s['taxa'] *= 100

# seaborn: paleta sequencial divergente, do vermelho ao verde.
sns.barplot(data=s, x='FX', y='taxa', hue='FX', legend=False, dodge=False, width=.6,
            palette=sns.color_palette('RdYlGn_r', len(s))[::-1], ax=ax)
for i, v in enumerate(s['taxa']):
    ax.text(i, v + 1.5, virgula(v, 1, '%'), ha='center', fontweight='bold', fontsize=10)

ax.set(xlabel='', ylabel='Taxa de detratores', ylim=(0, 80))
ax.yaxis.set_major_formatter(fmt(0, '%'))
plt.setp(ax.get_xticklabels(), rotation=20)

# matplotlib: eixo secundario para o NPS, que varia de negativo a positivo.
ax2 = ax.twinx()
ax2.grid(False)
sns.lineplot(x=range(len(s)), y=s['nps'], marker='o', color=AZ_ESC, lw=2.4, markersize=7, ax=ax2)
ax2.axhline(0, color=CINZA, ls=':', lw=1.2)
ax2.set(ylabel='NPS médio', ylim=(-60, 60))
ax2.yaxis.set_major_formatter(fmt(0))
ax2.yaxis.label.set_color(AZ_ESC)
ax2.tick_params(axis='y', colors=AZ_ESC)

ax.set_title('Cancelamento: efeito da antecedência do aviso sobre a detração', pad=12)
n_cancel = f'{len(c):,}'.replace(',', '.')
ax.text(0, -.26, f'Apenas voos cancelados (n = {n_cancel}). Barras: taxa de detratores. '
        'Linha: NPS médio.', transform=ax.transAxes, fontsize=8.5, color='#555')
mostrar(fig, 'g8_antecedencia_cancelamento')

**Leitura.** É o resultado de maior magnitude identificado na exploração. Para o **mesmo evento negativo**, a antecedência do aviso produz uma amplitude de dezenas de pontos percentuais na detração e mais de oitenta pontos de NPS.

O padrão é monotônico e a maior parte do efeito concentra-se nos primeiros quinze dias. A partir de 30 dias a curva estabiliza em patamar próximo à média geral da base, o que sugere que **o cancelamento comunicado com antecedência suficiente deixa de ser um evento de detração**.

Cabe a ressalva de que a antecedência **não é aleatória**. Cancelamentos de mesmo dia decorrem tipicamente de causas operacionais agudas, que carregam transtorno adicional além da falta de aviso. A separação entre o efeito da comunicação e o efeito da causa exigiria controle adicional, e fica registrada como hipótese a validar.

### Gráfico 4. Taxa de detratores por tier de fidelidade e faixa de atraso

*Tipo:* mapa de calor. *Variáveis:* `TIER_VIAGEM`, `FAIXA_ATRASO` e taxa de detratores.

O mapa de calor é a forma adequada aqui porque a pergunta é sobre **interação**, não sobre efeitos marginais. Duas variáveis categóricas cruzadas com uma medida contínua é exatamente o caso de uso do `seaborn.heatmap`.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))

TIERS = ['Sem cadastro', 'Azul Fidelidade', 'Topazio', 'Safira', 'Diamante']
tiers = [t for t in TIERS if t in set(df['TIER_VIAGEM'].dropna())]

h = (df[df['TIER_VIAGEM'].isin(tiers)]
     .pivot_table(index='TIER_VIAGEM', columns='FAIXA_ATRASO', values='DETRATOR',
                  aggfunc='mean', observed=True)
     .reindex(index=tiers, columns=ORD_ATRASO) * 100)

sns.heatmap(h, annot=rotulos(h, 1), fmt='', cmap='RdYlBu_r', vmin=8, vmax=85,
            linewidths=.6, linecolor='white', ax=ax,
            annot_kws={'fontweight': 'bold', 'fontsize': 10.5},
            cbar_kws={'label': 'Taxa de detratores (%)'})
ax.collections[0].colorbar.ax.yaxis.set_major_formatter(fmt(0))
ax.set_xticklabels(LAB_ATRASO, rotation=0, fontsize=9.5)
ax.set(xlabel='', ylabel='')
ax.set_title('Taxa de detratores por tier de fidelidade e faixa de atraso', pad=12)
ax.text(0, -.24, 'A penalização por atraso cresce com o tier: o Cliente Diamante '
        'detrata mais que o sem cadastro em todas as faixas.',
        transform=ax.transAxes, fontsize=8.5, color='#555')
mostrar(fig, 'g3_heatmap_tier_atraso')

**Leitura.** O mapa revela uma **interação entre fidelização e falha operacional** que não seria visível em análises marginais. Em voos pontuais o Cliente Diamante detrata sensivelmente mais que o Cliente sem cadastro. Em voos com mais de 120 minutos de atraso, os dois grupos convergem para o mesmo patamar elevado.

O padrão é consistente com o princípio de que a expectativa de serviço cresce com o nível de relacionamento: **o Cliente mais fidelizado é o menos tolerante à falha**. Para a modelagem, isso indica que `TIER_VIAGEM` e `FAIXA_ATRASO` não devem ser tratadas apenas como efeitos aditivos. Modelos baseados em árvores capturam essa interação naturalmente, enquanto uma regressão logística exigiria termo de interação explícito.

### Gráfico 5. Sazonalidade da detração, controlada por faixa de atraso

*Tipo:* pequenos múltiplos com séries de linha paralelas. *Variáveis:* mês do ano, `FAIXA_ATRASO` e taxa de detratores.

A pergunta metodológica é se a variação mensal decorre apenas da operação ou se há componente sazonal próprio. Os pequenos múltiplos do `seaborn.relplot` resolvem isso por construção: cada painel já é um controle por faixa de atraso.

In [ ]:
d = df.copy()
d['Mês'] = d['DATA_STD'].dt.month
d['Faixa'] = pd.Categorical(d['FAIXA_ATRASO'], categories=ORD_ATRASO, ordered=True)
agr = d.groupby(['Faixa', 'Mês'], observed=True)['DETRATOR'].mean().mul(100).reset_index()

# seaborn: relplot cuida da facetagem, um painel por faixa de atraso.
g = sns.relplot(data=agr, x='Mês', y='DETRATOR', col='Faixa', hue='Faixa', kind='line',
                marker='o', lw=2.2, markersize=5, palette=PALETA_AZUL, legend=False,
                facet_kws={'sharey': False}, height=3.4, aspect=.95)

# matplotlib: preenchimento sob a curva e rotulos de mes em cada painel.
meses = ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D']
for ax, chave, titulo, cor in zip(g.axes.flat, ORD_ATRASO, LAB_ATRASO, PALETA_AZUL):
    sub = agr[agr['Faixa'] == chave]
    ax.fill_between(sub['Mês'], sub['DETRATOR'].min() - 1, sub['DETRATOR'], color=cor, alpha=.12)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(meses, fontsize=8)
    ax.set_title(titulo.replace('\n', ' '), fontsize=10, fontweight='bold')
    ax.yaxis.set_major_formatter(fmt(0, '%'))
    ax.set_xlabel('')

g.axes.flat[0].set_ylabel('Taxa de detratores')
g.figure.suptitle('Sazonalidade da detração, controlada por faixa de atraso',
                  fontweight='bold', y=1.05)
g.figure.text(.09, -.06, 'Eixo vertical livre por painel: o interesse é a forma da curva, '
              'não o nível absoluto.', fontsize=8.5, color='#555')
mostrar(g.figure, 'g9_sazonalidade')

**Leitura.** O padrão de dezembro alto e agosto baixo **persiste dentro de todas as quatro faixas**, inclusive entre voos sem atraso algum. A diferença, portanto, não pode ser atribuída à pontualidade.

A hipótese explicativa combina composição de passageiro, com alta concentração de viajantes de lazer e de primeira viagem no período de férias, e congestionamento de infraestrutura, que afeta a experiência sem se traduzir em atraso registrado. A sazonalidade deve ser incorporada como **covariável**, e não tratada como ruído.

O eixo vertical é deliberadamente livre entre painéis (`sharey=False`): o que se compara aqui é a **forma** da curva dentro de cada faixa, não o nível, que já é conhecido do Gráfico 1.

### Gráfico 6. Correlação entre variáveis operacionais e a detração

*Tipo:* matriz de correlação de Spearman, triangular inferior. *Variáveis:* oito numéricas, incluindo o alvo binarizado.

Spearman e não Pearson: todas as numéricas apresentam forte assimetria positiva, como a seção 5.1 mostrou, e o alvo é binário. A máscara triangular evita a redundância da matriz simétrica.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))

cols = ['DETRATOR', 'ESTATISTICA_ATRASOSAIDA', 'ATRASO_CHEGADA', 'TEMPO_VOO',
        'N_TRECHOS', 'QTDE_VIAGENS_12M', 'QTDE_VIAGENS_24M', 'QTDE_VIAGENS_36M']
corr = df[cols].corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(corr, mask=mask, annot=rotulos(corr, 2), fmt='', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=.6, linecolor='white', ax=ax,
            annot_kws={'fontsize': 9, 'fontweight': 'bold'},
            cbar_kws={'label': 'Correlação de Spearman', 'shrink': .8})
ax.collections[0].colorbar.ax.yaxis.set_major_formatter(fmt(2))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
ax.set_title('Correlação entre variáveis operacionais e a detração', pad=12)
mostrar(fig, 'g5_correlacao')

print('Correlacao de cada variavel com o alvo, em ordem decrescente:')
print(corr['DETRATOR'].drop('DETRATOR').sort_values(ascending=False).round(3).to_string())

**Leitura.** A matriz confirma que **nenhuma variável operacional isolada apresenta correlação forte com a detração**. Combinado com os valores de V de Cramér da seção 5.4, o resultado sustenta que a detração é fenômeno multivariado e que a escolha de um classificador não linear se justifica pela ausência de preditor dominante.

O fato de o atraso na chegada superar o atraso na saída é coerente com a experiência do Cliente, já que o custo percebido do atraso se materializa no destino e não no portão de embarque. A observação deve ser lida com a ressalva da seção 6.2: a regra de cálculo de `ATRASO_CHEGADA` ainda aguarda validação do parceiro, e parte da associação pode decorrer da inclusão de tempo de reacomodação em voos cancelados.

Destacam-se dois blocos de colinearidade. O primeiro, entre os dois campos de atraso. O segundo, mais severo, entre `QTDE_VIAGENS_12M`, `_24M` e `_36M`, o que exigirá selecionar apenas uma das janelas ou construir uma razão entre elas. Há ainda associação entre `TEMPO_VOO` e `N_TRECHOS`, esperada por construção.

### Exportação das figuras

As sete figuras foram gravadas em disco com os mesmos nomes que `documents/documentacao.md` referencia em `assets/`. A célula abaixo confere o que foi escrito.

In [ ]:
if DIR_FIGURAS and os.path.isdir(DIR_FIGURAS):
    for arquivo in sorted(os.listdir(DIR_FIGURAS)):
        caminho = os.path.join(DIR_FIGURAS, arquivo)
        print(f'{arquivo:<40} {os.path.getsize(caminho) / 1024:>7.0f} KB')
    print(f'\nTotal em {os.path.abspath(DIR_FIGURAS)}')
else:
    print('DIR_FIGURAS desativado: as figuras foram apenas exibidas.')

# No Colab, para baixar as sete de uma vez:
# import shutil
# from google.colab import files
# shutil.make_archive('figuras', 'zip', DIR_FIGURAS)
# files.download('figuras.zip')

Basta descompactar o resultado em `assets/`, na raiz do repositório, para que as imagens da seção 4.2.1 passem a resolver.

---

## 10. Síntese e implicações para as próximas fases

1. **A base é de alta qualidade estrutural.** Junção 1:1 completa entre as três tabelas transacionais, ausência de duplicidades e cobertura temporal de 36 meses.
2. **A amostra não é representativa da população de passageiros.** O viés está correlacionado com o alvo, e a pós-estratificação com a tabela fornecida pela Azul reconcilia o NPS da amostra com a métrica oficial da companhia.
3. **A operação explica a maior parte da detração, mas não toda.** O piso irredutível em condições operacionais ideais delimita o teto de desempenho realista de um modelo baseado em variáveis de operação.
4. **Dois achados são acionáveis independentemente do modelo:** o limiar de 20 a 30 minutos como janela de maior retorno para intervenção, e a antecedência do aviso de cancelamento como fator de mitigação de magnitude elevada.

### Restrições que seguem para a modelagem

| Restrição | Origem |
|---|---|
| Partição agrupada por `ID_GOLDENRECORD` | Dependência intracliente, seção 3.1 |
| Validação com partição temporal | Efeito de período, seção 8.4 |
| Usar apenas uma das três janelas de `QTDE_VIAGENS` | Colinearidade, Gráfico 6 |
| Substituir `SUB_FIL_FREQUENCIAAZUL` por `QTDE_VIAGENS_12M` | Erro de medida, seção 6.3 |
| Excluir `N_ASSENTOS_AUDIT` do conjunto de preditores | Redundância com `N_TRECHOS`, seção 7.2 |
| Excluir os campos `NPS_*` de etapa da jornada | Vazamento de dados, seção 4 |
| Tratar `TIER_VIAGEM` e `FAIXA_ATRASO` com interação | Gráfico 4 |

Permanece **uma pendência técnica aberta com o parceiro**: a validação da regra de cálculo de `ATRASO_CHEGADA`.

---

### Referências

CHAPMAN, P. et al. **CRISP-DM 1.0: step-by-step data mining guide**. SPSS Inc., 2000.

CRAMÉR, H. **Mathematical methods of statistics**. Princeton: Princeton University Press, 1946.

GROVES, R. M.; PEYTCHEVA, E. The impact of nonresponse rates on nonresponse bias: a meta-analysis. **Public Opinion Quarterly**, v. 72, n. 2, p. 167-189, 2008.

HUNTER, J. D. Matplotlib: a 2D graphics environment. **Computing in Science & Engineering**, v. 9, n. 3, p. 90-95, 2007.

McKINNEY, W. Data structures for statistical computing in Python. In: **Proceedings of the 9th Python in Science Conference**, p. 56-61, 2010.

REICHHELD, F. F. The one number you need to grow. **Harvard Business Review**, v. 81, n. 12, p. 46-54, 2003.

VALLIANT, R. Post-stratification and conditional variance estimation. **Journal of the American Statistical Association**, v. 88, n. 421, p. 89-96, 1993.

WASKOM, M. L. Seaborn: statistical data visualization. **Journal of Open Source Software**, v. 6, n. 60, 3021, 2021.